# 🔧 LangChain Tools — Complete Access Context
### State + Context + Store + Stream Writer in one notebook
> **Python 3.11 | LangChain v1.0.0 | .env config**


## Cell 1 — Install & Imports

In [ ]:
%pip install langchain-openai python-dotenv langgraph -q

from dataclasses import dataclass
from typing import Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from langgraph.graph import MessagesState
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

## Cell 2 — LLM Setup

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)
print("✅ LLM configured!")

## Cell 3 — Overview: ToolRuntime Components

| Component       | Access via              | What it does                              | Mutable? |
|-----------------|-------------------------|-------------------------------------------|----------|
| **State**       | `runtime.state`         | Short-term memory for current conversation| ✅ Yes   |
| **Context**     | `runtime.context`       | Immutable config passed at invoke time    | ❌ No    |
| **Store**       | `runtime.store`         | Long-term memory across conversations     | ✅ Yes   |
| **Stream Writer**| `runtime.stream_writer`| Emit real-time progress updates           | —        |
| **Tool Call ID**| `runtime.tool_call_id`  | Unique ID for current tool invocation     | —        |

All of these are accessed via the `runtime: ToolRuntime` parameter —
which is **always hidden from the LLM**.


---
## 🟦 PART 1 — State (`runtime.state`)
Short-term memory. Exists only for the current conversation.
Read with `runtime.state["key"]`, update with `Command(update={...})`.


## Cell 4 — State Tools: Read & Update

In [ ]:
# READ state
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content
    return "No user messages found"


@tool
def get_user_preference(pref_name: str, runtime: ToolRuntime) -> str:
    """Get a user preference value by name."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")


# UPDATE state
@tool
def set_user_name(new_name: str) -> Command:
    """Set the user's name in the conversation state."""
    print(f"   ✏️  Updating state: user_name = '{new_name}'")
    return Command(update={"user_name": new_name})


print("✅ State tools defined!")
print(f"   LLM sees for get_user_preference   : {list(get_user_preference.args_schema.schema().get('properties',{}).keys())}")
print(f"   LLM sees for get_last_user_message : {list(get_last_user_message.args_schema.schema().get('properties',{}).keys())}")

## Cell 5 — State: Custom Schema & Agent

In [ ]:
class AgentState(MessagesState):
    user_name        : str
    user_preferences : dict

state_agent = create_agent(
    llm,
    tools=[get_last_user_message, get_user_preference, set_user_name],
    state_schema=AgentState,
    system_prompt="You are a helpful assistant. Use tools to read and update user info."
)
print("✅ State agent created!")

## Cell 6 — State: Run Examples

In [ ]:
# READ: last message
print("─" * 50)
print("🟦 READ — last user message from state")
print("─" * 50)
r = state_agent.invoke({
    "messages"       : [HumanMessage(content="Hello! I love Python."),
                        HumanMessage(content="What was my last message?")],
    "user_name"      : "",
    "user_preferences": {},
})
print("🤖", r["messages"][-1].content)

# READ: custom field
print("\n" + "─" * 50)
print("🟦 READ — preference from custom state field")
print("─" * 50)
r = state_agent.invoke({
    "messages"        : [HumanMessage(content="What is my theme preference?")],
    "user_name"       : "",
    "user_preferences": {"theme": "dark", "language": "Python"},
})
print("🤖", r["messages"][-1].content)

# UPDATE: set name
print("\n" + "─" * 50)
print("🟦 UPDATE — set user_name via Command")
print("─" * 50)
r = state_agent.invoke({
    "messages"        : [HumanMessage(content="My name is Priya. Please remember it.")],
    "user_name"       : "",
    "user_preferences": {},
})
print("🤖", r["messages"][-1].content)
print(f"📌 state['user_name'] = '{r['user_name']}'")

---
## 🟩 PART 2 — Context (`runtime.context`)
Immutable config passed at invoke time. Perfect for user IDs, session IDs.
Access via `runtime.context.field_name`. Cannot be changed during a conversation.


## Cell 7 — Context: Schema & Tools

In [ ]:
USER_DATABASE = {
    "user123": {"name": "Alice Johnson", "account_type": "Premium",  "balance": 5000, "email": "alice@example.com"},
    "user456": {"name": "Bob Smith",     "account_type": "Standard", "balance": 1200, "email": "bob@example.com"},
    "user789": {"name": "Priya Sharma",  "account_type": "Premium",  "balance": 8750, "email": "priya@example.com"},
}

@dataclass
class UserContext:
    user_id    : str
    session_id : str = "default-session"

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id
    if user_id in USER_DATABASE:
        u = USER_DATABASE[user_id]
        return f"Name: {u['name']}\nType: {u['account_type']}\nBalance: ${u['balance']}"
    return "User not found."

@tool
def get_email(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's registered email address."""
    user_id = runtime.context.user_id
    if user_id in USER_DATABASE:
        return f"Email: {USER_DATABASE[user_id]['email']}"
    return "User not found."

context_agent = create_agent(
    llm,
    tools=[get_account_info, get_email],
    context_schema=UserContext,
    system_prompt="You are a financial assistant. Never ask for the user ID — it is already known."
)
print("✅ Context agent created!")

## Cell 8 — Context: Run Examples

In [ ]:
# Same question, different context = different user's data

print("─" * 50)
print("🟩 Alice (user123)")
print("─" * 50)
r = context_agent.invoke(
    {"messages": [HumanMessage(content="What is my balance?")]},
    context=UserContext(user_id="user123", session_id="sess-001")
)
print("🤖", r["messages"][-1].content)

print("\n" + "─" * 50)
print("🟩 Bob (user456)")
print("─" * 50)
r = context_agent.invoke(
    {"messages": [HumanMessage(content="What is my balance?")]},
    context=UserContext(user_id="user456", session_id="sess-002")
)
print("🤖", r["messages"][-1].content)

print("\n" + "─" * 50)
print("🟩 Priya (user789) — email")
print("─" * 50)
r = context_agent.invoke(
    {"messages": [HumanMessage(content="What email do I have on file?")]},
    context=UserContext(user_id="user789", session_id="sess-003")
)
print("🤖", r["messages"][-1].content)

---
## 🟨 PART 3 — Store (`runtime.store`)
Long-term memory. Persists across conversations.
Use `store.put()` to save, `store.get()` to read, `store.search()` to list all.


## Cell 9 — Store: Tools & Agent

In [ ]:
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info to long-term memory store."""
    runtime.store.put(("users",), user_id, user_info)
    print(f"   💾 Saved '{user_id}' to store")
    return f"Saved user info for '{user_id}'."

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info from long-term memory store."""
    item = runtime.store.get(("users",), user_id)
    if item:
        print(f"   📖 Found '{user_id}' in store")
        return str(item.value)
    return f"No info found for '{user_id}'."

store = InMemoryStore()

store_agent = create_agent(
    llm,
    tools=[save_user_info, get_user_info],
    store=store,
    system_prompt="You are a helpful assistant with long-term memory."
)
print("✅ Store agent created!")

## Cell 10 — Store: Session 1 (Save) & Session 2 (Retrieve)

In [ ]:
# SESSION 1 — save
print("─" * 50)
print("🟨 SESSION 1 — Save user")
print("─" * 50)
r = store_agent.invoke({"messages": [HumanMessage(
    content="Save: userid: abc123, name: Alice, age: 30, email: alice@example.com"
)]})
print("🤖", r["messages"][-1].content)

# SESSION 2 — new conversation, data still there
print("\n" + "─" * 50)
print("🟨 SESSION 2 — New conversation, retrieve persisted data")
print("─" * 50)
r = store_agent.invoke({"messages": [HumanMessage(
    content="Get user info for user with id 'abc123'"
)]})
print("🤖", r["messages"][-1].content)

# Direct store inspection
print("\n📦 Store contents:")
for item in store.search(("users",)):
    print(f"   {item.key} → {item.value}")

---
## 🟥 PART 4 — Stream Writer (`runtime.stream_writer`)
Emit real-time progress updates while the tool is running.
Use `agent.stream(..., stream_mode="custom")` to receive them.


## Cell 11 — Stream Writer: Tool & Agent

In [ ]:
import time

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    writer(f"🔍 Looking up data for city: {city}")
    time.sleep(0.3)
    writer(f"📡 Acquired data for city: {city}")
    time.sleep(0.3)
    writer(f"✅ Done!")

    return f"It's always sunny in {city}!"

@tool
def generate_report(month: str, runtime: ToolRuntime) -> str:
    """Generate a sales report for a given month."""
    writer = runtime.stream_writer

    writer(f"📂 Starting report for: {month}")
    time.sleep(0.3)
    writer(f"🔍 Fetching sales data...")
    time.sleep(0.3)
    writer(f"📊 Calculating totals...")
    time.sleep(0.3)
    writer(f"✅ Report ready!")

    return f"Sales Report — {month}: Revenue $1.2M, Growth +8%"

stream_agent = create_agent(
    llm,
    tools=[get_weather, generate_report],
    system_prompt="You are a helpful assistant."
)
print("✅ Stream agent created!")

## Cell 12 — Stream Writer: Run with invoke() vs stream()

In [ ]:
# invoke() — no live updates
print("─" * 50)
print("🟥 invoke() — no streaming (only final answer)")
print("─" * 50)
r = stream_agent.invoke({"messages": [HumanMessage(content="What is the weather in Mumbai?")]})
print("🤖", r["messages"][-1].content)
print("⚠️  No intermediate updates shown!\n")

# stream() — live updates
print("─" * 50)
print("🟥 stream() — live updates visible")
print("─" * 50)
for chunk in stream_agent.stream(
    {"messages": [HumanMessage(content="What is the weather in Mumbai?")]},
    stream_mode="custom",
):
    print("📡 Live:", chunk)
print("✅ Done!")

## Cell 13 — Stream Writer: Capture Updates + Final Answer

In [ ]:
print("─" * 50)
print("🟥 stream_mode=['custom','updates'] — both live + answer")
print("─" * 50)

for chunk in stream_agent.stream(
    {"messages": [HumanMessage(content="Generate the sales report for March 2025")]},
    stream_mode=["custom", "updates"],
):
    mode, data = chunk
    if mode == "custom":
        print(f"📡 Live   : {data}")
    elif mode == "updates":
        msgs = data.get("agent", {}).get("messages", [])
        for msg in msgs:
            if hasattr(msg, "content") and msg.content:
                print(f"🤖 Answer : {msg.content}")

## ✅ Final Summary

| Component     | Access                   | Use for                          | Hidden from LLM? |
|---------------|--------------------------|----------------------------------|------------------|
| State         | `runtime.state["key"]`   | Conversation history, counters   | ✅ Yes           |
| Context       | `runtime.context.field`  | User ID, session ID              | ✅ Yes           |
| Store         | `runtime.store`          | Persistent cross-session memory  | ✅ Yes           |
| Stream Writer | `runtime.stream_writer`  | Real-time progress updates       | ✅ Yes           |
| Tool Call ID  | `runtime.tool_call_id`   | Correlate tool calls             | ✅ Yes           |

### Update State → `Command`
```python
return Command(update={"field": value})
```

### Update Store → `store.put()`
```python
runtime.store.put(("namespace",), key, value)
```

### Stream Updates → `writer()`
```python
writer = runtime.stream_writer
writer("progress message...")
# receive with: agent.stream(..., stream_mode="custom")
```
